# 05 — Doppler and the Range-Doppler Map


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vinculum3141-ship-it/active-radar-tracker-basics/blob/radar-tracker-notebooks/beginner/05-doppler-range-doppler.ipynb)


## What this notebook teaches

So far you have measured *where* a target is (range) with the matched filter. A radar can also measure *how fast* the target is moving toward or away from you. That measurement is Doppler, and it turns the pulses you already send into a velocity reading.

By the end of this notebook, you should be able to explain:

- the difference between fast time and slow time,
- why a moving target's echo phase advances from pulse to pulse,
- how an FFT across pulses turns that phase advance into a Doppler frequency,
- how the Doppler frequency maps to a radial velocity,
- and why the baseline 40 m/s target wraps to a wrong speed.

Keep these five questions in mind as you work through the cells. A dedicated section at the end answers each one directly.


## Setup and baseline values

The shared helpers give you one waveform (the chirp), the matched filter, and the baseline radar parameters. This notebook adds three new ideas on top: fast time, slow time, and the Doppler phase they reveal.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/vinculum3141-ship-it/active-radar-tracker-basics.git"
BRANCH_NAME = "radar-tracker-notebooks"
REPO_DIR = Path("/content/active-radar-tracker-basics")

in_colab = "google.colab" in sys.modules

if in_colab and not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--branch", BRANCH_NAME, "--single-branch", REPO_URL, str(REPO_DIR)],
        check=True,
    )

if in_colab:
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)

    repo_root = os.path.abspath(".")
    if repo_root not in sys.path:
        sys.path.insert(0, repo_root)

    print(f"Ready in Colab from {REPO_DIR}")
else:
    repo_root = os.path.dirname(os.getcwd())
    if repo_root not in sys.path:
        sys.path.insert(0, repo_root)
    print("Running locally; the repository is already available in this workspace.")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from beginner.helpers import baseline_spec
from beginner.helpers import (
    lfm_chirp,
    matched_filter,
    delay_samples_for_range,
    range_from_delay_samples,
)
from beginner.helpers.doppler import (
    doppler_frequency_hz,
    velocity_from_doppler,
    build_pulse_stack,
    range_doppler_map,
)
from beginner.helpers.math import wavelength_m
from beginner.helpers.plotting import apply_notebook_style

np.random.seed(42)
apply_notebook_style()
radar_spec = baseline_spec()
radar_spec


## Where we are in the story

Notebook 04 ended with a matched filter that pulls a noisy echo into a sharp peak. That peak tells you the echo's *delay*, which is range. But a single peak cannot tell you whether the target is moving. To learn velocity, you need many pulses, so you can watch the echo's phase change from one pulse to the next.

This notebook is the first major milestone: it turns a batch of 64 ordinary pulses into a two-dimensional picture of range *and* velocity.


## Fast time versus slow time

Within a single PRI, you sample the receive window rapidly at 20 MHz. That is **fast time**: the axis you used in every earlier notebook, where a sample index maps to a time-of-arrival and hence a range.

But a radar transmits one pulse per PRI, thousands of times per second. If you collect the echoes of many pulses and index them by *which* pulse they came from, you get a second, much slower clock: **slow time**. Each slow-time tick is one PRI, or 1 ms here.

Two targets can be hard to separate in fast time (they overlap in range) and easy to separate in slow time (their phases rotate at different rates). That is the point of this notebook: range lives in fast time, velocity lives in slow time.


## The pulse stack is a two-dimensional grid

Putting many pulses together makes a 2D array. It is worth being precise about its two axes, because everything from here reuses them.

- **Rows are slow time.** Each row is one pulse, indexed by pulse number. Slow time advances once per PRI — that is, once per pulse, at the 1000 Hz PRF. The row index is what you will FFT across to get Doppler.
- **Columns are fast time.** Each column is one range bin — a particular fast-time sample inside the receive window, mapped to a distance. The column index carries range.

So a **range bin** is a fast-time column (a range cell), while **once per PRI** is how the slow-time *rows* are sampled. They are two different axes, not two names for the same thing. A single cell of the stack, at (pulse k, range bin r), is one sampled value of the echo at that range, on that pulse.

One more thing must be said about that value: it is now a **complex number**. In earlier notebooks you worked with the real waveform as it might appear on a single wire. To see Doppler you must track the carrier's *phase*, which needs both the in-phase and quadrature parts — the real and imaginary components. The chirp you built in Notebook 01 is already complex, so each cell carries a real part and an imaginary part. That is why the stack is built `dtype=complex`: the phase, the thing that moves, only exists if you keep both parts.

## Build a pulse stack: the echo phase advances

A moving target at range R produces an echo that arrives at the same fast-time delay in every PRI (over a short observation it barely moves). What changes is the *phase* of that echo: each round trip is a little shorter (or longer) because the target moved, so the carrier wave returns at a slightly different phase.

Here we build the received signal for all 64 pulses. Every pulse gets the same delayed echo, but with a phase that rotates by 2*pi*fd*PRI from one pulse to the next, where fd is the Doppler frequency. The result is a **pulse stack**: 64 rows, one per pulse, each row a fast-time buffer.


In [ ]:
# Build the chirp and the per-pulse Doppler phase rotation, by hand.
pulse_len = int(round(radar_spec.pulse_width_s * radar_spec.fs_hz))
chirp = lfm_chirp(pulse_len, radar_spec.bandwidth_hz, radar_spec.pulse_width_s, radar_spec.fs_hz)

n_delay = delay_samples_for_range(radar_spec.target_range_m, radar_spec.fs_hz)
attenuation_linear = 10.0 ** (-40.0 / 20.0)

# Carrier wavelength and the Doppler frequency for a 20 m/s approach.
lam = wavelength_m(radar_spec.fc_hz)
velocity_mps = 20.0
fd_hz = 2.0 * velocity_mps / lam

n_pulses = radar_spec.n_pulses
pri_s = radar_spec.pri_s

fast_len = n_delay + pulse_len
stack = np.zeros((n_pulses, fast_len), dtype=complex)
for k in range(n_pulses):
    phase = np.exp(1j * 2.0 * np.pi * fd_hz * k * pri_s)
    stack[k, n_delay : n_delay + pulse_len] = attenuation_linear * chirp * phase

print(f"lambda      = {lam:.4f} m")
print(f"Doppler for {velocity_mps:.0f} m/s = {fd_hz:.1f} Hz")
print(f"Stack shape = {stack.shape}  ({n_pulses} pulses x {fast_len} fast-time samples)")


## What the phase advance looks like

The plot shows the echo at the target's fast-time range bin extracted from every pulse. As slow time advances (pulse index on the x-axis), the complex echo swings through a sinusoid at the Doppler frequency. A stationary target would sit still; a moving target rotates.

That rotation is the unit of Doppler. Counting how fast the phase goes around is how you recover velocity.


In [ ]:
# Extract the complex echo at the target's range bin across all pulses.
# The raw echo occupies samples n_delay .. n_delay + pulse_len; we take a point
# in the middle of it. (The matched filter will later concentrate this to the
# single peak you derived in Notebook 04.)
range_bin = n_delay + pulse_len // 2
slow_time_samples = stack[:, range_bin]  # one complex number per pulse

pulse_idx = np.arange(n_pulses)
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(pulse_idx, slow_time_samples.real, color="#1b9e77", label="Real part")
ax.plot(pulse_idx, slow_time_samples.imag, color="#d95f02", label="Imaginary part")
ax.set_xlabel("Pulse index (slow time)")
ax.set_ylabel("Echo at target range bin")
ax.set_title(f"Echo phase advances across slow time at {fd_hz:.0f} Hz (target {velocity_mps:.0f} m/s)")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

print(f"The echo completes about {fd_hz * pri_s * n_pulses:.1f} full rotations over {n_pulses} pulses.")


## Compress every pulse into a range profile

Notebook 04's matched filter turns one noisy echo into a sharp peak. Applying it to *every* pulse in the stack gives 64 range profiles - one per pulse. Stacking them keeps the same shape as the pulse stack, but now each fast-time sample is a compressed range bin rather than raw signal.

Range resolution now lives in fast time (the matched-filter peak's width), while Doppler lives in slow time. The range-Doppler map combines both.


In [ ]:
# Matched-filter every pulse to get a stack of range profiles.
profiles = np.array([matched_filter(stack[k], chirp) for k in range(n_pulses)])
print(f"Range profiles shape = {profiles.shape}")

# The matched-filter output is longer than the fast-time buffer (full mode).
peak_bin = np.argmax(np.abs(profiles[0]))
print(f"Compressed peak at fast-time bin {peak_bin} (delay = {peak_bin - (pulse_len - 1)} samples)")


## The FFT across pulses: from phase to velocity

The echo's phase advances at fd cycles per second. If you take the fast Fourier transform of the slow-time samples at one range bin, you get a peak at exactly fd. That is the Doppler spectrum: a bump sitting at the target's Doppler frequency.

Each slow-time tick is one PRI, so the **slow-time sampling rate is the PRF** (1000 Hz here). An FFT across pulses therefore measures frequencies from -PRF/2 to +PRF/2, as you saw in Notebook 01 for a time-domain signal, but now sampled once per PRI.

To turn a Doppler frequency into a speed, invert fd = 2v/lambda: v = fd * lambda / 2.


In [ ]:
# FFT the slow-time samples at the target's range bin.
nfft = 512  # zero-pad for a smooth spectrum
slow_fft = np.fft.fftshift(np.fft.fft(slow_time_samples, nfft))
doppler_axis = np.fft.fftshift(np.fft.fftfreq(nfft, pri_s))

peak_fd = doppler_axis[np.argmax(np.abs(slow_fft))]
peak_vel = velocity_from_doppler(peak_fd, radar_spec.fc_hz)

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(doppler_axis, np.abs(slow_fft), color="#7570b3", linewidth=1.0)
ax.axvline(peak_fd, color="#d95f02", linestyle="--", linewidth=1)
ax.set_xlabel("Doppler frequency (Hz)")
ax.set_ylabel("|FFT|")
ax.set_title("Doppler spectrum at the target range bin")
plt.tight_layout()
plt.show()

print(f"Detected Doppler = {peak_fd:.1f} Hz  ->  velocity {peak_vel:.1f} m/s (true {velocity_mps:.0f} m/s)")


## The range-Doppler map

Doing the slow-time FFT at *every* fast-time bin, not just the target's, produces a two-dimensional map: range along one axis, velocity along the other. Each cell's brightness is the echo power for that range-and-velocity combination.

The result is a heatmap with a single bright blob at the target's range and velocity. This is how a pulse radar displays a scene: one glance shows where targets are *and* how fast they are moving, and separate blobs mean separate targets that might overlap in either axis alone.


In [ ]:
# Slow-time FFT at every range bin -> the range-Doppler map.
_, velocity_axis, rdm = range_doppler_map(profiles, pri_s, radar_spec.fc_hz)

range_axis = range_from_delay_samples(
    np.arange(profiles.shape[1]) - (pulse_len - 1), radar_spec.fs_hz
)

fig, ax = plt.subplots(figsize=(9, 5))
im = ax.imshow(
    rdm,
    aspect="auto",
    origin="lower",
    extent=[range_axis[0], range_axis[-1], velocity_axis[0], velocity_axis[-1]],
    cmap="viridis",
)
ax.axhline(0, color="white", linewidth=0.6, linestyle=":")  # zero-velocity line
ax.set_xlabel("Range (m)")
ax.set_ylabel("Velocity (m/s)")
ax.set_title("Range-Doppler map: one target at 1000 m, 20 m/s")
plt.colorbar(im, ax=ax, label="|Echo power|")
plt.tight_layout()
plt.show()


## Velocity resolution and ambiguity

Just as in range, the Doppler estimate has limits. Two of them matter here.

The radar processes a batch of pulses together as one coherent group. That batch is the **CPI** (coherent processing interval), and its length is set by how many pulses you collect: here the baseline uses N = 64 pulses (the `n_pulses` setting), so one CPI lasts N*PRI = 64 ms.

**Velocity resolution.** Within one CPI the slow-time FFT has 64 samples (one per pulse), so it can tell apart frequencies separated by about one bin. That becomes a velocity resolution of delta_v = lambda / (2 N PRI) about 0.96 m/s for our 64 pulses. Two targets whose speeds differ by less than that look like one.

**Velocity ambiguity.** Slow time samples once per PRI, at the PRF of 1000 Hz. Just as sampling fast time at fs limits the highest frequency you can name, sampling slow time at the PRF limits the highest Doppler you can name uniquely to +-PRF/2 = +-500 Hz, i.e. +-30.6 m/s. A target faster than that **aliases** to a wrong, lower speed.

The baseline target moves at 40 m/s - faster than 30.6 m/s. Its Doppler of 654 Hz is beyond 500 Hz, so it folds over and is reported at the wrong velocity. Let's see it.


In [ ]:
# Baseline target moves at 40 m/s, which exceeds the unambiguous limit.
fast_velocity_mps = radar_spec.target_velocity_mps  # 40 m/s
fd_fast = doppler_frequency_hz(fast_velocity_mps, radar_spec.fc_hz)

stack_fast = build_pulse_stack(chirp, n_pulses, pri_s, fd_fast, n_delay, attenuation_linear)
profiles_fast = np.array([matched_filter(stack_fast[k], chirp) for k in range(n_pulses)])
_, velocity_axis_fast, rdm_fast = range_doppler_map(profiles_fast, pri_s, radar_spec.fc_hz)

peak_v = velocity_axis_fast[np.unravel_index(np.argmax(rdm_fast), rdm_fast.shape)[0]]

print(f"True velocity     = {fast_velocity_mps:.0f} m/s  -> Doppler {fd_fast:.0f} Hz")
print(f"Unambiguous limit = +/-{np.max(np.abs(velocity_axis_fast)):.1f} m/s")
print(f"Reported velocity = {peak_v:.1f} m/s")

fig, ax = plt.subplots(figsize=(9, 5))
im = ax.imshow(
    rdm_fast,
    aspect="auto",
    origin="lower",
    extent=[range_axis[0], range_axis[-1], velocity_axis_fast[0], velocity_axis_fast[-1]],
    cmap="viridis",
)
ax.axhline(0, color="white", linewidth=0.6, linestyle=":")
ax.set_xlabel("Range (m)")
ax.set_ylabel("Velocity (m/s)")
ax.set_title("Aliasing: 40 m/s target reported as moving the wrong way")
plt.colorbar(im, ax=ax, label="|Echo power|")
plt.tight_layout()
plt.show()


## Checkpoint

In your own words, what is the difference between fast time and slow time, and which one carries range versus velocity?

Then answer this: if a target's Doppler frequency doubles, what happens to its reported velocity?


## Common mistake

A common mistake is to confuse fast-time sampling with slow-time sampling. Fast time is sampled at fs = 20 MHz inside one PRI and builds the range axis. Slow time is sampled once per PRI at the PRF and builds the velocity axis. The two clocks do not see the same frequencies, and mixing them up hides the velocity story.

Another mistake is to treat the raw pulse stack's magnitude as the Doppler signal. The echo's *magnitude* stays roughly constant across pulses; it is the *phase* that rotates. You must look at the complex value - its real and imaginary parts, or its phase - to see Doppler at all.


## Why the helpers exist

The cells above built the pulse stack, compressed each pulse, and ran the slow-time FFT step by step so you can see where Doppler comes from. Once the idea is clear, the same work collapses into build_pulse_stack and range_doppler_map, so later notebooks can stand up a range-Doppler scene in a few lines instead of two dozen.

Keep the first pass visible for the physics, then use the helpers when the lesson shifts elsewhere.


In [ ]:
# The same scene in helper form: build and map in a few lines.
stack_h = build_pulse_stack(chirp, n_pulses, pri_s, fd_hz, n_delay, attenuation_linear)
profiles_h = np.array([matched_filter(stack_h[k], chirp) for k in range(n_pulses)])
_, v_h, rdm_h = range_doppler_map(profiles_h, pri_s, radar_spec.fc_hz)

peak_v_h = v_h[np.unravel_index(np.argmax(rdm_h), rdm_h.shape)[0]]
print(f"Helper-based velocity estimate = {peak_v_h:.1f} m/s (true {velocity_mps:.0f} m/s)")


## Stretch: two targets, one range, two velocities

Two targets at the *same range* but different velocities are indistinguishable in a single matched filter - they produce one peak in fast time. Doppler separates them. Add a second target at 1000 m moving at -20 m/s (receding) alongside the 20 m/s one, build the summed stack, and confirm the range-Doppler map shows two distinct blobs split along the velocity axis.


In [ ]:
# Two targets at the same range, opposite velocities.
vel_a = 20.0
vel_b = -20.0

stack2 = build_pulse_stack(chirp, n_pulses, pri_s, doppler_frequency_hz(vel_a, radar_spec.fc_hz), n_delay, attenuation_linear)
stack2 += build_pulse_stack(chirp, n_pulses, pri_s, doppler_frequency_hz(vel_b, radar_spec.fc_hz), n_delay, attenuation_linear)

profiles2 = np.array([matched_filter(stack2[k], chirp) for k in range(n_pulses)])
_, v2, rdm2 = range_doppler_map(profiles2, pri_s, radar_spec.fc_hz)

fig, ax = plt.subplots(figsize=(9, 5))
im = ax.imshow(
    rdm2,
    aspect="auto",
    origin="lower",
    extent=[range_axis[0], range_axis[-1], v2[0], v2[-1]],
    cmap="viridis",
)
ax.axhline(0, color="white", linewidth=0.6, linestyle=":")
ax.set_xlabel("Range (m)")
ax.set_ylabel("Velocity (m/s)")
ax.set_title("Two targets at the same range, +20 m/s and -20 m/s")
plt.colorbar(im, ax=ax, label="|Echo power|")
plt.tight_layout()
plt.show()


## Closing the loop: answers to the opening questions

At the start we listed five things to be able to explain. Here is each answer.

**Fast time versus slow time.** Fast time is the rapid sampling inside one PRI (20 MHz) that resolves range. Slow time is the slower indexing across pulses (one sample per PRI, at the 1000 Hz PRF) that resolves velocity. Range is found in fast time; velocity is found in slow time.

**Why the echo phase advances.** Over one PRI the target moves a little, changing the round-trip distance and hence the carrier phase. Across pulses that phase advances steadily at the Doppler frequency fd = 2v/lambda. Its magnitude stays about constant; only the phase rotates.

**How the FFT turns that into Doppler.** Sampling the complex echo once per pulse (at the PRF) and taking an FFT across pulses produces a peak at fd. For a 20 m/s target at 2.45 GHz that peak sat at about 327 Hz.

**How Doppler maps to velocity.** Inverting fd = 2v/lambda gives v = fd * lambda / 2. The 327 Hz peak became about 20 m/s, matching the true speed.

**Why the 40 m/s target wraps.** Slow time samples at the 1000 Hz PRF, so the highest unambiguous Doppler is +-500 Hz, or +-30.6 m/s. The 40 m/s target wants 654 Hz, beyond the limit, so it folds to a lower frequency and the map reported it at about -21 m/s - the wrong speed and the wrong direction.

If you can retell these five answers, you have measured velocity - the second coordinate a pulse radar can report in addition to range.


## Summary

In this notebook you built on the matched filter to add a second measurement axis: velocity via Doppler. You learned that a pulse radar has two clocks - fast time for range within a PRI, and slow time for velocity across pulses. By building a 64-pulse stack, compressing each pulse, and taking an FFT along slow time, you produced a range-Doppler map with a single blob at the target's true range and velocity.

You also met the limits of that measurement. Velocity resolution is set by the CPI length and wavelength (about 0.96 m/s here), and the highest unambiguous velocity is set by half the PRF (30.6 m/s). The baseline 40 m/s target lives beyond that limit and aliased to a wrong speed - a concrete look at the ambiguity that real radars must manage with pulse repetition regimes.

Range and velocity are now both measurable. The next notebook takes the noisy range and velocity detections this map produces and smooths them over time into a stable track with a Kalman filter.
